# ⚡ V10 News Scraper (Industry-Enhanced)

**Features:**
- ✅ Concurrent sources (10 at once)
- ✅ RSS → Sitemap → Homepage discovery
- ✅ trafilatura + newspaper4k fallback
- ✅ A.I. → AI keyword fix

| Setting | Value |
|---------|-------|
| Keywords | 6 (with A.I.→AI) |
| Concurrent Sources | 10 |
| Max Matched/Source | 20 |
| Time Limit/Source | 60s |
| Date Limit | 14 days |

In [1]:
# CELL 1: IMPORTS
import asyncio
import aiohttp
import os
import json
import re
import hashlib
import time
from datetime import datetime, timedelta
from typing import List, Dict, Optional
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
import pandas as pd
from tqdm.notebook import tqdm

import trafilatura
import feedparser
import nest_asyncio
nest_asyncio.apply()

# newspaper4k for fallback extraction
try:
    from newspaper import Article
    NEWSPAPER_AVAILABLE = True
except ImportError:
    NEWSPAPER_AVAILABLE = False
    print("⚠️ newspaper4k not installed - using trafilatura only")

print(f"✅ Ready (newspaper4k: {NEWSPAPER_AVAILABLE})")

✅ Ready (newspaper4k: True)


In [2]:
# CELL 2: CONFIG
class CFG:
    SOURCES_PATH = "sources.json"
    OUTPUT_PATH = "scraped_news_v10.csv"
    
    # Parallelism
    CONCURRENT_SOURCES = 10          # Process 10 sources at once
    
    # Limits
    TARGET_MATCHES_PER_SOURCE = 20
    TIME_LIMIT_PER_SOURCE = 60
    DATE_LIMIT_DAYS = 14
    MIN_CONTENT = 200
    TIMEOUT = 10
    
    # 6 FIXED KEYWORDS
    KEYWORDS = ["A.I.", "Google", "Microsoft", "Nvidia", "Crypto", "Chipset"]
    
    USER_AGENT = "Mozilla/5.0 Chrome/120.0.0.0"
    
    COLUMNS = ["source", "headline", "author", "url", "published",
               "content_snippet", "url_hash", "full_content", "method", "scraped_at"]

print(f"⚙️ Keywords: {CFG.KEYWORDS}")
print(f"⚙️ {CFG.CONCURRENT_SOURCES} concurrent sources")
print(f"⏱️ {CFG.TARGET_MATCHES_PER_SOURCE} matches/source, {CFG.TIME_LIMIT_PER_SOURCE}s limit")

⚙️ Keywords: ['A.I.', 'Google', 'Microsoft', 'Nvidia', 'Crypto', 'Chipset']
⚙️ 10 concurrent sources
⏱️ 20 matches/source, 60s limit


In [3]:
# CELL 3: UTILS
def clean_text(text):
    if not text: return ""
    text = BeautifulSoup(text, 'html.parser').get_text(' ')
    return re.sub(r'\s+', ' ', text).strip()

def matches_keywords(text):
    """V5/V6/V7 style: A.I. also matches AI."""
    if not text: return []
    text_lower = text.lower()
    matched = []
    
    for kw in CFG.KEYWORDS:
        kw_lower = kw.lower()
        if kw_lower == "a.i.":
            if "a.i." in text_lower or re.search(r'\bai\b', text_lower):
                matched.append(kw)
        else:
            if kw_lower in text_lower:
                matched.append(kw)
    return matched

def parse_date(date_str):
    if not date_str: return None
    for fmt in ['%Y-%m-%dT%H:%M:%S', '%Y-%m-%d', '%a, %d %b %Y %H:%M:%S', '%a, %d %b %Y']:
        try:
            return datetime.strptime(str(date_str)[:25].replace(' GMT', '').replace(' +0000', ''), fmt)
        except: pass
    return None

def is_too_old(date_str):
    dt = parse_date(date_str)
    if not dt: return False
    return dt < datetime.now() - timedelta(days=CFG.DATE_LIMIT_DAYS)

print("✅ Utils (A.I.→AI fix applied)")

✅ Utils (A.I.→AI fix applied)


In [4]:
# CELL 4: ENHANCED SCRAPER
class EnhancedScraper:
    def __init__(self):
        self.results = []
        self.seen = set()
        self.session = None
        self.lock = asyncio.Lock()
    
    async def get_session(self):
        if not self.session:
            connector = aiohttp.TCPConnector(limit=50)  # Connection pool
            self.session = aiohttp.ClientSession(
                connector=connector,
                timeout=aiohttp.ClientTimeout(total=CFG.TIMEOUT),
                headers={'User-Agent': CFG.USER_AGENT}
            )
        return self.session
    
    async def fetch(self, url):
        try:
            session = await self.get_session()
            async with session.get(url) as r:
                if r.status == 200:
                    return await r.text()
        except: pass
        return None
    
    # ==================== DISCOVERY ====================
    
    async def discover_rss(self, base_url):
        """Try to find RSS feed."""
        for path in ['/feed', '/rss', '/rss.xml', '/atom.xml', '/feed.xml', '/index.xml', '/news/feed']:
            html = await self.fetch(urljoin(base_url, path))
            if html and ('<rss' in html or '<feed' in html):
                try:
                    feed = feedparser.parse(urljoin(base_url, path))
                    if feed.entries:
                        return [{'url': e.get('link'), 'title': e.get('title', ''),
                                 'date': e.get('published', ''), 'method': 'rss'} 
                                for e in feed.entries]
                except: pass
        return []
    
    async def discover_sitemap(self, base_url):
        """Parse sitemap.xml for article URLs."""
        articles = []
        sitemap_paths = ['/sitemap.xml', '/sitemap_index.xml', '/news-sitemap.xml', 
                         '/sitemap-news.xml', '/post-sitemap.xml']
        
        for path in sitemap_paths:
            html = await self.fetch(urljoin(base_url, path))
            if not html or '<urlset' not in html and '<sitemapindex' not in html:
                continue
            
            try:
                soup = BeautifulSoup(html, 'xml')
                
                # Handle sitemap index (links to other sitemaps)
                for sitemap in soup.find_all('sitemap')[:3]:  # Max 3 sub-sitemaps
                    loc = sitemap.find('loc')
                    if loc:
                        sub_html = await self.fetch(loc.text)
                        if sub_html:
                            sub_soup = BeautifulSoup(sub_html, 'xml')
                            for url_tag in sub_soup.find_all('url')[:50]:
                                loc = url_tag.find('loc')
                                if loc:
                                    articles.append({'url': loc.text, 'title': '', 'date': '', 'method': 'sitemap'})
                
                # Direct URL list
                for url_tag in soup.find_all('url')[:50]:
                    loc = url_tag.find('loc')
                    if loc:
                        articles.append({'url': loc.text, 'title': '', 'date': '', 'method': 'sitemap'})
                
                if articles:
                    break
            except: pass
        
        return articles
    
    async def scan_homepage(self, base_url):
        """Fallback: scan homepage for article links."""
        articles = []
        html = await self.fetch(base_url)
        if not html: return []
        
        soup = BeautifulSoup(html, 'html.parser')
        domain = urlparse(base_url).netloc
        seen = set()
        
        for a in soup.find_all('a', href=True):
            href = urljoin(base_url, a['href'])
            p = urlparse(href)
            if p.netloc != domain or href in seen or len(p.path) < 15:
                continue
            seen.add(href)
            articles.append({'url': href, 'title': a.get_text(strip=True)[:100], 'date': '', 'method': 'homepage'})
        return articles
    
    # ==================== EXTRACTION ====================
    
    def extract_with_trafilatura(self, html):
        """Primary extraction with trafilatura."""
        try:
            result = trafilatura.bare_extraction(html)
            if result:
                return {
                    'title': result.get('title', ''),
                    'author': result.get('author', ''),
                    'content': clean_text(result.get('text', ''))
                }
        except: pass
        return None
    
    def extract_with_newspaper(self, url, html):
        """Fallback extraction with newspaper4k."""
        if not NEWSPAPER_AVAILABLE:
            return None
        try:
            article = Article(url)
            article.set_html(html)
            article.parse()
            if article.text and len(article.text) >= CFG.MIN_CONTENT:
                return {
                    'title': article.title or '',
                    'author': ', '.join(article.authors) if article.authors else '',
                    'content': clean_text(article.text)
                }
        except: pass
        return None
    
    # ==================== PROCESSING ====================
    
    async def process_article(self, article):
        url = article.get('url')
        if not url:
            return None
        
        async with self.lock:
            if url in self.seen:
                return None
            self.seen.add(url)
        
        html = await self.fetch(url)
        if not html:
            return None
        
        # Try trafilatura first, then newspaper4k fallback
        extracted = self.extract_with_trafilatura(html)
        if not extracted or len(extracted.get('content', '')) < CFG.MIN_CONTENT:
            extracted = self.extract_with_newspaper(url, html)
        
        if not extracted or len(extracted.get('content', '')) < CFG.MIN_CONTENT:
            return None
        
        content = extracted['content']
        
        # Keyword check
        kws = matches_keywords(content)
        if not kws:
            return None
        
        return {
            'source': urlparse(url).netloc,
            'headline': extracted.get('title') or article.get('title', ''),
            'author': extracted.get('author', ''),
            'url': url,
            'published': article.get('date', ''),
            'content_snippet': content[:200],
            'url_hash': hashlib.md5(url.encode()).hexdigest(),
            'full_content': content,
            'method': article['method'],
            'scraped_at': datetime.now().isoformat()
        }
    
    async def process_source(self, source):
        """Discovery: RSS → Sitemap → Homepage. Smart stopping."""
        url = source['url']
        start_time = time.time()
        matches = 0
        
        # Discovery waterfall: RSS → Sitemap → Homepage
        articles = await self.discover_rss(url)
        if not articles:
            articles = await self.discover_sitemap(url)
        if not articles:
            articles = await self.scan_homepage(url)
        
        if not articles:
            return 0
        
        # Process with smart stopping
        for article in articles:
            if matches >= CFG.TARGET_MATCHES_PER_SOURCE:
                break
            if time.time() - start_time > CFG.TIME_LIMIT_PER_SOURCE:
                break
            if is_too_old(article.get('date')):
                break
            
            result = await self.process_article(article)
            if result:
                async with self.lock:
                    self.results.append(result)
                matches += 1
        
        return matches
    
    def save(self):
        if self.results:
            pd.DataFrame(self.results, columns=CFG.COLUMNS).to_csv(CFG.OUTPUT_PATH, index=False)
            print(f"💾 Saved {len(self.results)} articles")
    
    async def close(self):
        if self.session:
            await self.session.close()

print("✅ EnhancedScraper Ready")

✅ EnhancedScraper Ready


In [5]:
# CELL 5: MAIN (CONCURRENT SOURCES)
async def main():
    with open(CFG.SOURCES_PATH) as f:
        sources = json.load(f)['sources']
    
    print(f"🚀 V10 Enhanced | {len(sources)} sources | {CFG.CONCURRENT_SOURCES} concurrent")
    print(f"🔑 Keywords: {CFG.KEYWORDS}")
    print(f"📋 Discovery: RSS → Sitemap → Homepage")
    print(f"🔧 Extraction: trafilatura → newspaper4k fallback")
    print()
    
    scraper = EnhancedScraper()
    
    # Process sources in batches (concurrent)
    semaphore = asyncio.Semaphore(CFG.CONCURRENT_SOURCES)
    
    async def process_with_semaphore(src):
        async with semaphore:
            return await scraper.process_source(src)
    
    # Create tasks with progress bar
    with tqdm(total=len(sources), desc="Scraping") as pbar:
        tasks = []
        for src in sources:
            task = asyncio.create_task(process_with_semaphore(src))
            task.add_done_callback(lambda _: pbar.update(1))
            tasks.append(task)
        
        results = await asyncio.gather(*tasks, return_exceptions=True)
        pbar.set_postfix({'total': len(scraper.results)})
    
    await scraper.close()
    scraper.save()
    
    # Summary
    methods = {}
    for r in scraper.results:
        m = r.get('method', 'unknown')
        methods[m] = methods.get(m, 0) + 1
    
    print(f"\n🏁 Done! {len(scraper.results)} articles")
    print(f"📊 Methods: {methods}")

await main()

🚀 V10 Enhanced | 50 sources | 10 concurrent
🔑 Keywords: ['A.I.', 'Google', 'Microsoft', 'Nvidia', 'Crypto', 'Chipset']
📋 Discovery: RSS → Sitemap → Homepage
🔧 Extraction: trafilatura → newspaper4k fallback



Scraping:   0%|          | 0/50 [00:00<?, ?it/s]

💾 Saved 348 articles

🏁 Done! 348 articles
📊 Methods: {'rss': 311, 'homepage': 31, 'sitemap': 6}


In [6]:
# CELL 6: ANALYTICS
if os.path.exists(CFG.OUTPUT_PATH):
    df = pd.read_csv(CFG.OUTPUT_PATH)
    print(f"📊 Total: {len(df)} articles")
    print(f"\n📈 By Method:")
    print(df['method'].value_counts().to_string())
    print(f"\n🏆 Top Sources:")
    print(df['source'].value_counts().head(10).to_string())

📊 Total: 348 articles

📈 By Method:
method
rss         311
homepage     31
sitemap       6

🏆 Top Sources:
source
www.tomsguide.com        20
9to5google.com           20
decrypt.co               20
www.coindesk.com         20
www.zdnet.com            19
techcrunch.com           18
www.techradar.com        18
www.tomshardware.com     18
analyticsindiamag.com    15
spectrum.ieee.org        13
